# M1_210210 with birth-cloud dust, and with the dust index down to −3

- off: `dust1_off`; current notebook defaults, diffuse Kriek & Conroy dust only.
- on: `dust1_on`; birth-cloud power law, $\tau_{\rm bc} = r_{\rm dust}\,\tau_{\rm dust}$, index −1, ages ≤ 10 Myr.
- m3: `dust_index_m3`; $\delta_{\rm dust}$ prior Uniform(−3, 0.4).
- Same seed 20260832, data and NSS settings in all three.
- Metallicity is $[\mathrm{Fe}/\mathrm{H}]$ = grid $Z$ + `FEH_OFFSET`.

In [ ]:
import os
import re
import sys
from pathlib import Path

os.environ.setdefault("JAX_PLATFORMS", "cpu")
import corner
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("CERIDWEN_PROJECT_ROOT", Path.cwd()))
while not (PROJECT_ROOT / "scripts" / "per_galaxy_diagnostics.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import per_galaxy_diagnostics as pgd  # noqa: E402
from build_dr2_quiescent_summary import FEH_OFFSET, formation_times  # noqa: E402
from spectral_figures import (  # noqa: E402
    mark_absorption_features,
    mark_rest_wavelength_axis,
    set_plain_log_ticks,
    spectral_tight_layout,
)

RESULTS = PROJECT_ROOT / "results/birth-cloud-dust"
TARGET = "M1_210210"
FIT_DIRS = {"off": RESULTS / "dust1_off" / f"210210-{TARGET}",
            "on": RESULTS / "dust1_on" / f"210210-{TARGET}",
            "m3": RESULTS / "dust_index_m3" / f"210210-{TARGET}"}
COLOURS = {"off": "#222222", "on": "#0072B2", "m3": "#D55E00"}
NAMES = {"off": "birth-cloud dust off", "on": "birth-cloud dust on", "m3": r"$\delta_{\rm dust} \geq -3$"}
REF = "off"
LABELS = {"logmass": r"$\log_{10}(M_\star/M_\odot)$", "Z": r"$[\mathrm{Fe}/\mathrm{H}]$", "afe": r"$[\alpha/\mathrm{Fe}]$",
          "tau_dust": r"$\tau_{\mathrm{dust}}$", "dust_index": r"$\delta_{\rm dust}$", "age": r"$t_{\mathrm{MW}}$ [Gyr]"}
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.precision", 3)

## Parameters, evidence and $\chi^2$

- `spec_chi2_raw`: catalogue uncertainties only, posterior-median model.
- `spec_chi2_fref`: every fit at the dust1_off fit's calibration floor.
- `phot_chi2` and `phot_n` refer to each fit's own bands.
- `u_pull`: CFHT $u^*$ (data − model) / uncertainty at the posterior median.

In [ ]:
def _text(v):
    return v.decode() if isinstance(v, bytes) else str(v)


def load_fit(folder):
    out = {"folder": folder}
    with h5py.File(folder / "ceridwen_derived_outputs.h5", "r") as d:
        out["attrs"] = dict(d.attrs)
        names = [_text(v) for v in d["summary/parameter"][:]]
        out["summary"] = pd.DataFrame({"q16": d["summary/q16"][:], "q50": d["summary/q50"][:], "q84": d["summary/q84"][:]}, index=names)
        out["diag"] = dict(d["diagnostics"].attrs)
        for grp in ("spectrum", "photometry", "sfh", "calibration"):
            out[grp] = {k: d[grp][k][:] for k in d[grp]}
        out["calibration"]["attrs"] = dict(d["calibration"].attrs)
    with h5py.File(folder / "ceridwen_result.h5", "r") as r:
        out["tau_max"] = float(re.search(r"diffuse_tau_kc: .*?Uniform\(0, ([0-9.]+)\)", _text(r["model"].attrs["parameter_block"])).group(1))
        out["n_calls"] = int(r["samples"].attrs["n_likelihood_calls"])
        out["wall_s"] = float(r["samples"].attrs["wall_time_s"])
    out["galaxy"] = pgd.load_galaxy(folder)
    return out


fits = {k: load_fit(v) for k, v in FIT_DIRS.items()}
REF_FLOOR = fits[REF]["summary"].loc["calibration floor [%]", "q50"]


def chi2_at_floor(fit, f_pct):
    spec = fit["spectrum"]; mask = spec["mask"].astype(bool)
    sigma = np.hypot(spec["uncertainty"], (f_pct / 100.0) * np.abs(spec["posterior_q50"]))
    return float(np.sum(((spec["observed"] - spec["posterior_q50"]) / sigma)[mask] ** 2))


def summary_row(fit):
    s, d = fit["summary"], fit["diag"]
    g = fit["galaxy"]; w = pgd.posterior_weights(g)
    t20, t50, t80 = formation_times(fit["sfh"]["lookback_time_gyr"], fit["sfh"]["mass_fraction_draws"])
    row = dict(order=int(fit["calibration"]["attrs"]["order"]), tau_max=fit["tau_max"], seed=g.seed, lnZ=float(d["log_evidence"]), lnZ_err=float(d["log_evidence_err"]),
               n_pix=int(fit["spectrum"]["mask"].sum()), spec_chi2_raw=chi2_at_floor(fit, 0.0), spec_chi2_fref=chi2_at_floor(fit, REF_FLOOR),
               spec_chi2_stored=float(d["spectrum_chi2"]), phot_chi2=float(d["photometry_chi2"]), phot_n=int(d["photometry_ndof"]),
               ess=float(d["posterior_weight_ess"]), passed=bool(d["passed"]), calls=fit["n_calls"], wall_s=fit["wall_s"])
    for name, key in [("logmass", "logmass"), ("Z", "Z"), ("afe", "afe"), ("tau_dust", "diffuse_tau_kc"),
                      ("age", "mass-weighted age [Gyr]"), ("f_calib", "calibration floor [%]")]:
        row[f"{name}_q50"], row[f"{name}_hw"] = s.loc[key, "q50"], 0.5 * (s.loc[key, "q84"] - s.loc[key, "q16"])
    row["feh_q50"], row["feh_hw"] = row.pop("Z_q50") + FEH_OFFSET, row.pop("Z_hw")
    q16, q50, q84 = (pgd.weighted_quantile(np.ravel(g.samples["diffuse_dust_index"]), w, q) for q in (0.16, 0.5, 0.84))
    row["dust_index_q50"], row["dust_index_hw"] = q50, 0.5 * (q84 - q16)
    for name, t in [("t20", t20), ("t50", t50), ("t80", t80)]:
        q16, q50, q84 = np.percentile(t, [16, 50, 84])
        row[f"{name}_q50"], row[f"{name}_hw"] = q50, 0.5 * (q84 - q16)
    phot = fit["photometry"]
    row["u_pull"] = float(phot["pull"][[_text(f) for f in phot["filters"]].index("cfht_megacam_us_9301")])
    if "dust_ratio" in g.samples:
        q16, q50, q84 = (pgd.weighted_quantile(np.ravel(g.samples["dust_ratio"]), w, q) for q in (0.16, 0.5, 0.84))
        row["dust_ratio_q50"], row["dust_ratio_hw"] = q50, 0.5 * (q84 - q16)
    for key in ("zred", "sigma_smooth"):
        if key in s.index:
            row[f"{key}_q50"], row[f"{key}_hw"] = s.loc[key, "q50"], 0.5 * (s.loc[key, "q84"] - s.loc[key, "q16"])
    return row


table = pd.DataFrame({k: summary_row(v) for k, v in fits.items()}).T
table.to_csv(RESULTS / "comparison.csv")
display(table.T)

## SFH and corner

- SFH median with 16-84 band; corner uses weighted dead points, 1σ contours.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for k, fit in fits.items():
    sfh = fit["sfh"]; lb = sfh["lookback_time_gyr"]
    axes[0].plot(lb, sfh["normalized_sfr_q50"], color=COLOURS[k], lw=1.4, label=NAMES[k])
    axes[0].fill_between(lb, sfh["normalized_sfr_q16"], sfh["normalized_sfr_q84"], color=COLOURS[k], alpha=0.12, lw=0)
    younger = np.concatenate([[0.0], np.cumsum(sfh["mass_fraction_q50"])])
    axes[1].plot(lb, younger / younger[-1], color=COLOURS[k], lw=1.4, label=NAMES[k])
axes[0].set(xscale="log", yscale="log", xlabel="lookback time [Gyr]", ylabel=r"SFR / $M_{\rm formed}$ [yr$^{-1}$]", xlim=(0.02, lb[-1] * 1.05))
axes[1].set(xlabel="lookback time [Gyr]", ylabel="mass fraction younger", ylim=(0, 1.02))
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(RESULTS / f"sfh-{TARGET}.png", dpi=130); plt.show()

In [ ]:
CORNER_PARAMS = ["logmass", "Z", "afe", "tau_dust", "dust_index", "age"]


def mw_age_samples(g):
    r = np.asarray(g.samples["logsfr_ratios"])
    h = 10.0 ** np.concatenate([np.zeros((len(r), 1)), -np.cumsum(r, axis=1)], axis=1)
    edges = g.sfh_edges_gyr
    m = 0.5 * (h[:, :-1] + h[:, 1:]) * np.diff(edges)
    return (m * (0.5 * (edges[:-1] + edges[1:]))).sum(1) / m.sum(1)


def corner_data(g):
    return np.column_stack([g.samples["logmass"], g.samples["Z"] + FEH_OFFSET, g.samples["afe"], g.samples["diffuse_tau_kc"],
                            g.samples["diffuse_dust_index"], mw_age_samples(g)])


data = {k: (corner_data(f["galaxy"]), pgd.posterior_weights(f["galaxy"])) for k, f in fits.items()}
n = len(CORNER_PARAMS)
lo = np.min([[pgd.weighted_quantile(x[:, i], w, 0.002) for i in range(n)] for x, w in data.values()], axis=0)
hi = np.max([[pgd.weighted_quantile(x[:, i], w, 0.998) for i in range(n)] for x, w in data.values()], axis=0)
rng_ = [(a, b) if b > a else (a - 1e-3, a + 1e-3) for a, b in zip(lo, hi)]
fig = None
for k, (x, w) in data.items():
    fig = corner.corner(x, weights=w, labels=[LABELS[p] for p in CORNER_PARAMS], range=rng_, color=COLOURS[k], fig=fig, bins=40, smooth=1.0,
                        plot_datapoints=False, plot_density=False, fill_contours=False, levels=[1 - np.exp(-0.5)],
                        hist_kwargs=dict(density=True, lw=1.4), contour_kwargs=dict(linewidths=1.4))
fig.legend(handles=[plt.Line2D([], [], color=COLOURS[k], label=NAMES[k]) for k in fits], loc="upper right", frameon=False, fontsize=9)
fig.suptitle(TARGET, y=0.98)
fig.savefig(RESULTS / f"corner-{TARGET}.png", dpi=150); plt.show()